In [3]:
import pandas as pd
import numpy as np
import math
from collections import Counter
import re
import textstat

def tokenize(text): #this tokenizer will split words based on the aplabetical content, 
    if not text: # "def tokenize(text): boom1 bang" -> print(tokenize("def tokenize(text): boom1 bang"))
        return []
    return re.findall(r"[A-Za-z_][A-Za-z0-9_]*", text.lower())

def calculate_entropy(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return np.nan 
    words = tokenize(text)
    if not words:
        return np.nan
    counts = Counter(words)
    probs = [c / len(words) for c in counts.values()]
    return -sum(p * math.log2(p) for p in probs)

def strip_comments(text):
    # 1. Remove multi-line comments /* ... */
    text = re.sub(r'/\*.*?\*/', '', text, flags=re.DOTALL)
    
    # 2. Split into lines to handle inline comments
    lines = text.splitlines()
    clean_lines = []
    
    for line in lines:
        # Remove // or # style comments
        # This regex looks for // or # and grabs everything until the end of the line
        line = re.sub(r'(//|#).*$', '', line)
        
        # Remove triple quote docstrings if they are on a single line 
        # (e.g., """ doc """)
        line = re.sub(r'(""".*?"""|' + "'''.*?''')", '', line)
        
        # Only keep the line if it isn't empty after stripping whitespace
        if line.strip():
            clean_lines.append(line.rstrip())
            
    return "\n".join(clean_lines)

def doc_code_overlap(doc_text, code_text): # get percent of tokens that overlap between code and text
    doc_tokens = set(tokenize(doc_text))
    code_tokens = set(tokenize(code_text))

    if not doc_tokens or not code_tokens:
        return np.nan
    overlap = doc_tokens.intersection(code_tokens)
    return len(overlap) / len(doc_tokens)

def doc_redundancy(doc_text): # get percent of tokens that are repeated within the documentation
    tokens = tokenize(doc_text)
    if not tokens:
        return np.nan
    
    unique = len(set(tokens))
    return 1 - (unique / len(tokens))

In [ ]:
import pandas as pd
import numpy as np
import math
from collections import Counter
import re
import textstat

output_path = r"G:\On the Naturalness of Agent-Generated Documentation\dataset\data\test.csv"
df = pd.read_csv(r"G:\On the Naturalness of Agent-Generated Documentation\dataset\data\agent_final_dataset_subset_b.csv")
df_human = pd.read_csv(r"G:\On the Naturalness of Agent-Generated Documentation\dataset\data\dev_final_dataset_subset_b.csv")

df["group"] = "agent"
df_human["group"] = "human"
df = pd.concat([df, df_human], ignore_index=True)


def clean_comments(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # 1. Extract Block/Docstring Content
    # Using [/*]{2,3} handles both /* and /**
    blocks = re.findall(r'/\*+([\s\S]*?)\*/', text)
    docstrings = re.findall(r'["\']{3}([\s\S]*?)["\']{3}', text)
    
    # 2. Extract Single-Line
    inline = re.findall(r'(?:#|///|//)\s*(.*?)\s*(?=#|///|//|$)', text)

    combined = blocks + docstrings + inline
    cleaned_comments = []

    for item in combined:
        # Remove the leading '*' from each line (common in JSDoc/C-style)
        # This regex looks at the start of lines and removes whitespace + '*'
        clean_item = re.sub(r'^\s*\* ?', '', item, flags=re.MULTILINE)
        
        # Collapse newlines and tabs into a single space
        clean_item = ' '.join(clean_item.split())
        
        if clean_item.strip():
            cleaned_comments.append(clean_item.strip())
                   
    return ' '.join(cleaned_comments)


# Output: ['This is a docstring', 'This is an inline block', 'This is a single line comment', 'Another comment here']

documentation = {}

for i,a in zip(df['doc_text'], df['label']):
    if pd.isna(a):
        a = 'human'
    if a in documentation:
        documentation[a].append(clean_comments(i))
    else:
        documentation[a] = [clean_comments(i)]


for idx, row in df.iterrows():
    print(row["doc_text"])
    doc_text = clean_comments(row["doc_text"])
    print(doc_text)
    print("\n\n-------------------------\n\n")

    code_text = row["function"]
    code_text_no_doc = strip_comments(code_text)
    if isinstance(doc_text, str) and doc_text.strip():
        
        df.at[idx, "doc_entropy"] = calculate_entropy(doc_text)
        df.at[idx, "doc_readability"] = textstat.flesch_reading_ease(doc_text) if doc_text else np.nan
        df.at[idx, "doc_code_overlap"] = doc_code_overlap(doc_text, code_text_no_doc)
        df.at[idx, "doc_redundancy"] = doc_redundancy(doc_text)
    else:
        # For missing or empty docs, keep NaN
        df.at[idx, "doc_entropy"] = np.nan
        df.at[idx, "doc_readability"] = np.nan
        df.at[idx, "doc_code_overlap"] = np.nan
        df.at[idx, "doc_redundancy"] = np.nan


# Save the updated dataset

df.to_csv(output_path, index=False)
print("Recomputed metrics and saved to updated_dataset_2_metrics.csv")

df = pd.read_csv(output_path)
df = df[~df['file_path'].str.contains(r'\.next/', na=False)]
df.to_csv(output_path, index=False)

## If not merged already, merge in the new agent data

In [ ]:
# ── 9. Merge new_agent_dataset.csv + final_dataset.csv → final_dataset_new.csv ──
NEW_AGENT_OUTPUT_PATH = r"G:\On the Naturalness of Agent-Generated Documentation\dataset\data\new_agent_dataset.csv"


df_original = pd.read_csv(output_path)
df_new      = pd.read_csv(NEW_AGENT_OUTPUT_PATH)

print(f'final_dataset.csv:     {df_original.shape}')
print(f'new_agent_dataset.csv: {df_new.shape}')
print()
print('Group breakdown in original dataset:')
print(df_original['group'].value_counts().to_string())

# Align columns before concat (handles any ordering differences)
df_new_aligned = df_new.reindex(columns=df_original.columns)

df_merged = pd.concat([df_original, df_new_aligned], ignore_index=True)

# Deduplicate on the natural key — file_path must be included so that
# anonymous functions in different files of the same PR are not collapsed.
before    = len(df_merged)
df_merged = df_merged.drop_duplicates(
    subset=['repo', 'pull_request', 'file_path', 'function_name', 'function_start_line'],
    keep='first'
).reset_index(drop=True)

if before - len(df_merged):
    print(f'Dropped {before - len(df_merged)} duplicate rows.')

print()
print(f'Merged dataset shape: {df_merged.shape}')
print('Group breakdown in merged dataset:')
print(df_merged['group'].value_counts().to_string())
print('Label breakdown in merged dataset:')
print(df_merged['label'].value_counts().to_string())

df_merged.to_csv(output_path, index=False)
print(f'\nSaved to: {output_path}')

final_dataset.csv:     (13467, 30)
new_agent_dataset.csv: (531, 30)

Group breakdown in original dataset:
group
human    6918
agent    6549
Dropped 189 duplicate rows.

Merged dataset shape: (13809, 30)
Group breakdown in merged dataset:
group
agent    6975
human    6834
Label breakdown in merged dataset:
label
Claude_Code     2458
Copilot         1634
Cursor          1342
Devin            986
OpenAI_Codex     555

Saved to: G:\On the Naturalness of Agent-Generated Documentation\dataset\data\test.csv


In [ ]:
import pandas as pd
import numpy as np
import math
from collections import Counter
import re
import textstat


df = pd.read_csv(output_path)


def clean_comments(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # 1. Extract Block/Docstring Content
    # Using [/*]{2,3} handles both /* and /**
    blocks = re.findall(r'/\*+([\s\S]*?)\*/', text)
    docstrings = re.findall(r'["\']{3}([\s\S]*?)["\']{3}', text)
    
    # 2. Extract Single-Line
    inline = re.findall(r'(?:#|///|//)\s*(.*?)\s*(?=#|///|//|$)', text)

    combined = blocks + docstrings + inline
    cleaned_comments = []

    for item in combined:
        # Remove the leading '*' from each line (common in JSDoc/C-style)
        # This regex looks at the start of lines and removes whitespace + '*'
        clean_item = re.sub(r'^\s*\* ?', '', item, flags=re.MULTILINE)
        
        # Collapse newlines and tabs into a single space
        clean_item = ' '.join(clean_item.split())
        
        if clean_item.strip():
            cleaned_comments.append(clean_item.strip())
                   
    return ' '.join(cleaned_comments)


for idx, row in df.iterrows():
    original_doc = row["doc_text"]
    code_text = row["function"]
    code_text_no_doc = strip_comments(code_text)

    doc_text = clean_comments(original_doc)

    if isinstance(doc_text, str) and doc_text.strip():
        # Extraction found comment markers — use cleaned text and update doc_text
        df.at[idx, "doc_entropy"] = calculate_entropy(doc_text)
        df.at[idx, "doc_readability"] = textstat.flesch_reading_ease(doc_text)
        df.at[idx, "doc_code_overlap"] = doc_code_overlap(doc_text, code_text_no_doc)
        df.at[idx, "doc_redundancy"] = doc_redundancy(doc_text)
        df.at[idx, "doc_text"] = doc_text
    elif isinstance(original_doc, str) and original_doc.strip():
        # No markers found — doc_text is already plain prose, compute metrics from it directly
        df.at[idx, "doc_entropy"] = calculate_entropy(original_doc)
        df.at[idx, "doc_readability"] = textstat.flesch_reading_ease(original_doc)
        df.at[idx, "doc_code_overlap"] = doc_code_overlap(original_doc, code_text_no_doc)
        df.at[idx, "doc_redundancy"] = doc_redundancy(original_doc)
        # doc_text stays unchanged
    else:
        # Genuinely no documentation
        df.at[idx, "doc_entropy"] = np.nan
        df.at[idx, "doc_readability"] = np.nan
        df.at[idx, "doc_code_overlap"] = np.nan
        df.at[idx, "doc_redundancy"] = np.nan


# Save the updated dataset
df.to_csv(output_path, index=False)
print("Recomputed metrics and saved to final_dataset_new.csv")

Recomputed metrics and saved to final_dataset_new.csv
